In [1]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import Ridge
from scipy.sparse import hstack


### 1. Загрузите данные об описаниях вакансий и соответствующих годовых зарплатах из файла salary-train.csv.


In [2]:
data_train = pd.read_csv('salary-train.csv')
data_test = pd.read_csv('salary-test-mini.csv')


### 2. Проведите предобработку: приведите тексты к нижнему регистру; замените все, кроме букв и цифр, на пробелы; примените TfidfVectorizer для преобразования текстов в векторы признаков, оставив только те слова, которые встречаются хотя бы в 5 объектах; замените пропуски в столбцах LocationNormalized и ContractTime на специальную строку ’nan’; примените DictVectorizer для получения one-hot-кодирования признаков LocationNormalized и ContractTime; объедините все полученные признаки в одну матрицу «объекты-признаки» с помощью scipy.sparse.hstack.


In [3]:
for data in (data_train, data_test):
    data['FullDescription'] = data['FullDescription'].str.lower().replace('[^a-zA-Z0-9]', ' ', regex=True)
    data['LocationNormalized'] = data['LocationNormalized'].fillna('nan')
    data['ContractTime'] = data['ContractTime'].fillna('nan')

vectorizer = TfidfVectorizer(min_df=5)
X_train_text = vectorizer.fit_transform(data_train['FullDescription'])
X_test_text = vectorizer.transform(data_test['FullDescription'])

encoder = DictVectorizer()
X_train_categ = encoder.fit_transform(data_train[['LocationNormalized', 'ContractTime']].to_dict('records'))
X_test_categ = encoder.transform(data_test[['LocationNormalized', 'ContractTime']].to_dict('records'))

X_train = hstack([X_train_text, X_train_categ])
X_test = hstack([X_test_text, X_test_categ])


### 3. Обучите гребневую регрессию с параметром alpha=1. Целевая переменная записана в столбце SalaryNormalized.


In [4]:
model = Ridge(alpha=1.0, random_state=241)
model.fit(X_train, data_train['SalaryNormalized'])


### 4. Постройте прогнозы для двух примеров из файла salary-test-mini.csv. Значения полученных прогнозов являются ответом на задание. Укажите их через пробел.


In [5]:
predictions = model.predict(X_test)
print(round(predictions[0], 2), round(predictions[1], 2))


56572.23 37191.87
